# Binary Classification Using a Single-Layer Perceptron
## Wine Recognition Dataset (Cultivar Class 0 vs Class 1)

**Dataset:** Wine Recognition Dataset, UCI Machine Learning Repository  
**Binary task:** Class 0 (`0`) vs Class 1 (`1`). Class 2 is excluded.  
**Samples used:** 130  
**Input features:** 13 chemical measurements.

This notebook implements the Single-Layer Perceptron from scratch using Python and NumPy. No prebuilt machine-learning classifier is used.

**Dataset source:** https://archive.ics.uci.edu/dataset/109/wine  
Keep `wine_binary_dataset.csv` in the same folder as this notebook.


## 1. Import libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path


## 2. Load the CSV dataset

In [ ]:
data_path = Path("wine_binary_dataset.csv")
if not data_path.exists():
    raise FileNotFoundError("Put wine_binary_dataset.csv in the same folder as this notebook.")

df = pd.read_csv(data_path)
display(df.head())
print("Dataset shape:", df.shape)
print("\nClass counts:")
print(df["class"].value_counts().sort_index())
print("\nMissing values:")
print(df.isnull().sum())


## 3. Preprocess the data

In [ ]:
feature_columns = [
    "alcohol", "malic_acid", "ash", "alcalinity_of_ash", "magnesium",
    "total_phenols", "flavanoids", "nonflavanoid_phenols", "proanthocyanins",
    "color_intensity", "hue", "od280_od315", "proline"
]
X = df[feature_columns].to_numpy(dtype=float)
y = df["class"].to_numpy(dtype=int)

# Median imputation, if missing values are present.
for j in range(X.shape[1]):
    median = np.nanmedian(X[:, j])
    X[np.isnan(X[:, j]), j] = median

print("Samples:", len(X))
print("Features:", X.shape[1])
print("Labels:", np.unique(y))


## 4. Stratified 80/20 train-test split

In [ ]:
rng = np.random.default_rng(42)
train_indices, test_indices = [], []

for label in (0, 1):
    indices = np.where(y == label)[0]
    rng.shuffle(indices)
    n_test = int(round(0.20 * len(indices)))
    test_indices.extend(indices[:n_test])
    train_indices.extend(indices[n_test:])

train_indices = np.array(train_indices)
test_indices = np.array(test_indices)
rng.shuffle(train_indices)
rng.shuffle(test_indices)

X_train, X_test = X[train_indices], X[test_indices]
y_train, y_test = y[train_indices], y[test_indices]

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))


## 5. Standardize features

In [ ]:
# Fit scaling parameters on training data only to avoid test-data leakage.
mean = X_train.mean(axis=0)
std = X_train.std(axis=0)
std[std == 0] = 1.0

X_train = (X_train - mean) / std
X_test = (X_test - mean) / std


## 6. Single-Layer Perceptron implemented from scratch

In [ ]:
class SingleLayerPerceptron:
    def __init__(self, learning_rate=0.01, epochs=100, seed=42):
        self.learning_rate = learning_rate
        self.epochs = epochs
        self.seed = seed
        self.weights = None
        self.bias = 0.0

    @staticmethod
    def activation(z):
        return (z >= 0).astype(int)  # Binary step activation

    def predict(self, X):
        return self.activation(X @ self.weights + self.bias)

    def fit(self, X, y):
        n_samples, n_features = X.shape
        self.weights = np.zeros(n_features, dtype=float)
        self.bias = 0.0
        rng = np.random.default_rng(self.seed)

        for epoch in range(self.epochs):
            for i in rng.permutation(n_samples):
                z = np.dot(X[i], self.weights) + self.bias
                prediction = int(z >= 0)
                error = y[i] - prediction

                # Perceptron learning rule.
                self.weights += self.learning_rate * error * X[i]
                self.bias += self.learning_rate * error
        return self


## 7. Train and test the model

In [ ]:
model = SingleLayerPerceptron(learning_rate=0.01, epochs=100, seed=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print("Training completed.")


## 8. Evaluate model performance

In [ ]:
# Positive class is cultivar class 1.
TP = int(np.sum((y_test == 1) & (y_pred == 1)))
TN = int(np.sum((y_test == 0) & (y_pred == 0)))
FP = int(np.sum((y_test == 0) & (y_pred == 1)))
FN = int(np.sum((y_test == 1) & (y_pred == 0)))

accuracy = (TP + TN) / len(y_test)
precision = TP / (TP + FP) if TP + FP else 0.0
recall = TP / (TP + FN) if TP + FN else 0.0
f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0

print("========== SINGLE-LAYER PERCEPTRON RESULTS ==========")
print(f"Accuracy : {accuracy * 100:.2f}%")
print(f"Precision: {precision * 100:.2f}%")
print(f"Recall   : {recall * 100:.2f}%")
print(f"F1-score : {f1 * 100:.2f}%")

confusion = np.array([[TN, FP], [FN, TP]])
print("\nConfusion matrix (rows=actual, columns=predicted):")
print(confusion)


## 9. Plot the confusion matrix

In [ ]:
plt.figure(figsize=(6, 5))
plt.imshow(confusion, cmap="Blues")
plt.title("Single-Layer Perceptron - Confusion Matrix")
plt.xlabel("Predicted class")
plt.ylabel("Actual class")
plt.xticks([0, 1], ["Cultivar 0", "Cultivar 1"])
plt.yticks([0, 1], ["Cultivar 0", "Cultivar 1"])

for i in range(2):
    for j in range(2):
        plt.text(j, i, str(confusion[i, j]),
                 ha="center", va="center", fontsize=13)

plt.colorbar()
plt.tight_layout()
plt.savefig("wine_confusion_matrix.png", dpi=150)
plt.show()


## 10. Conclusion

A Single-Layer Perceptron was implemented from scratch with NumPy to classify two wine cultivars using 13 chemical measurements. The notebook reports accuracy, precision, recall, F1-score, and a confusion matrix on a held-out test set.

The original Wine Recognition dataset has three classes. This binary assignment uses only classes 0 and 1 and excludes class 2. The perceptron learns a linear decision boundary, so performance depends on the data split and model settings.

**Dataset reference:** Forina, M. et al. (1991). *Wine*. UCI Machine Learning Repository. https://doi.org/10.24432/C5PC7J
